**Week 9 — Multiple Linear Regression — สัมประสิทธิ์, t-statistic, Adjusted R², AIC/BIC, การเลือกตัวแปร, CI vs PI**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:**
  - ข้อมูลลูกค้า: `cust = pd.read_csv('biz_customers.csv')` (1203 แถว แถวละ 1 คน)
  - ข้อมูลแคมเปญ: `camp = pd.read_csv('biz_campaigns.csv')` (240 แถว แถวละ 1 แคมเปญ)
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์
- **แบบจำลอง Multiple Linear Regression (MLR):** ทุกโมเดลมี intercept และประมาณด้วย OLS ผ่าน `statsmodels.formula.api.smf.ols` หรือ `statsmodels.api.OLS`
- **เกณฑ์สารสนเทศ AIC และ BIC:** ใช้ค่าที่ statsmodels รายงาน (`model.aic`, `model.bic`)
- **ช่วงประมาณการ:**
  - Confidence Interval (CI) ของค่าเฉลี่ย: `get_prediction().summary_frame()['mean_ci_lower']` / `['mean_ci_upper']`
  - Prediction Interval (PI) ของรายตัว: `get_prediction().summary_frame()['obs_ci_lower']` / `['obs_ci_upper']`

### รายละเอียดไฟล์ข้อมูล

**1. `biz_customers.csv`** (1203 แถว)
- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `basket_size`: จำนวนสินค้าเฉลี่ยต่อออเดอร์
- `orders_90d`: จำนวนออเดอร์ใน 90 วันล่าสุด
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)

**2. `biz_campaigns.csv`** (240 แถว)
- `tv`: งบโฆษณาทางโทรทัศน์ (พันบาท)
- `radio`: งบโฆษณาทางวิทยุ (พันบาท)
- `social`: งบโฆษณาทางโซเชียลมีเดีย (พันบาท)
- `email`: งบโฆษณาทางอีเมล (พันบาท)
- `sales`: ยอดขายของแคมเปญ (พันชิ้น)

**โจทย์ข้อ 9.1:** ฝึกทำนาย `spend_k` ด้วย multiple linear regression จากตัวทำนาย `income_k`, `tenure_months`, `email_opens`, `web_visits`, `loyalty_member` (มี intercept) ใช้ลูกค้าทั้ง 1203 คนในไฟล์ `biz_customers.csv` ค่าสัมประสิทธิ์ของ `loyalty_member` = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลลูกค้าทั้งหมด
cust = pd.read_csv('biz_customers.csv')

# 2. Fit โมเดล Multiple Linear Regression
formula = 'spend_k ~ income_k + tenure_months + email_opens + web_visits + loyalty_member'
model = smf.ols(formula, data=cust).fit()
print(model.params)

# 3. ค่าสัมประสิทธิ์ของ loyalty_member
coef_loyalty = model.params['loyalty_member']
print(f"สัมประสิทธิ์ loyalty_member: {coef_loyalty:.4f}")
print(f"คำตอบข้อ 9.1: {coef_loyalty:.4f}")


Intercept         0.718392
income_k          0.048798
tenure_months     0.004704
email_opens       0.008287
web_visits        0.117491
loyalty_member    0.367504
dtype: float64
สัมประสิทธิ์ loyalty_member: 0.3675
คำตอบข้อ 9.1: 0.3675


**โจทย์ข้อ 9.2:** ฝึกก่อนใส่ `income_k` และ `basket_size` เป็นตัวทำนายร่วมกันในโมเดล ตรวจความสัมพันธ์ระหว่างสองคอลัมน์นี้ด้วยสัมประสิทธิ์สหสัมพันธ์เพียร์สัน (Pearson $r$) ของลูกค้าทั้ง 1203 คนใน `biz_customers.csv` $r$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลลูกค้า
cust = pd.read_csv('biz_customers.csv')

# 2. คำนวณ Pearson correlation ระหว่าง income_k กับ basket_size
corr_matrix = cust[['income_k', 'basket_size']].corr()
print("Correlation matrix:")
print(corr_matrix)

r = corr_matrix.loc['income_k', 'basket_size']
print(f"Pearson r: {r:.4f}")
print(f"คำตอบข้อ 9.2: {r:.4f}")


Correlation matrix:
             income_k  basket_size
income_k     1.000000     0.352225
basket_size  0.352225     1.000000
Pearson r: 0.3522
คำตอบข้อ 9.2: 0.3522


**โจทย์ข้อ 9.3:** ฝึกทำนาย `spend_k` จาก `income_k`, `web_visits`, `loyalty_member`, `age` (มี intercept) ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv` ค่า t-statistic ของสัมประสิทธิ์ `income_k` (ทดสอบ $H_0: \beta = 0$) = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลและ Fit โมเดล
cust = pd.read_csv('biz_customers.csv')
formula = 'spend_k ~ income_k + web_visits + loyalty_member + age'
model = smf.ols(formula, data=cust).fit()

# 2. หา t-statistic ของสัมประสิทธิ์ income_k
t_income = model.tvalues['income_k']
print(f"t-statistic ของ income_k: {t_income:.4f}")
print(f"คำตอบข้อ 9.3: {t_income:.2f}")


t-statistic ของ income_k: 17.5137
คำตอบข้อ 9.3: 17.51


**โจทย์ข้อ 9.4:** ฝึกใช้เฉพาะ**ลูกค้า 960 แถวแรก**ของไฟล์ `biz_customers.csv` (train) ทำนาย `spend_k` จาก `tenure_months`, `loyalty_member`, `basket_size`, `email_opens` (มี intercept) ค่า **Adjusted $R^2$** ของโมเดล = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. เลือก 960 แถวแรก (.iloc[:960])
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
print(f"ขนาดชุดข้อมูล Train: {train.shape}")

# 2. Fit โมเดล MLR
formula = 'spend_k ~ tenure_months + loyalty_member + basket_size + email_opens'
model = smf.ols(formula, data=train).fit()

# 3. ค่า Adjusted R2
adj_r2 = model.rsquared_adj
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {adj_r2:.4f}")
print(f"คำตอบข้อ 9.4: {adj_r2:.4f}")


ขนาดชุดข้อมูล Train: (960, 13)
R-squared: 0.0837
Adjusted R-squared: 0.0798
คำตอบข้อ 9.4: 0.0798


**โจทย์ข้อ 9.5:** ฝึกทำนาย `sales` จาก `tv`, `radio`, `social`, `email` (มี intercept) ใช้แคมเปญทั้ง 240 แคมเปญใน `biz_campaigns.csv` ค่า **BIC** ของโมเดล (ค่า `bic` ที่ statsmodels รายงาน) = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลแคมเปญทั้งหมด 240 แถว
camp = pd.read_csv('biz_campaigns.csv')

# 2. Fit โมเดลทำนาย sales
formula = 'sales ~ tv + radio + social + email'
model = smf.ols(formula, data=camp).fit()

# 3. ดึงค่า BIC
bic_val = model.bic
print(f"ค่า BIC ของโมเดล: {bic_val:.2f}")
print(f"คำตอบข้อ 9.5: {bic_val:.2f}")


ค่า BIC ของโมเดล: 1055.02
คำตอบข้อ 9.5: 1055.02


**โจทย์ข้อ 9.6:** กรณีศึกษาทีมวิเคราะห์ลูกค้าของร้านกาแฟสาขาหนึ่งต้องการโมเดลทำนายยอดซื้อต่อเดือน (`spend_k`) และมีตัวทำนายให้เลือก 5 ตัว ได้แก่ รายได้ (`income_k`), ระยะเวลาที่เป็นลูกค้า (`tenure_months`), การเป็นสมาชิกสะสมแต้ม (`loyalty_member`), อายุ (`age`), จำนวนอีเมลโปรโมชันที่เปิดอ่าน (`email_opens`) ทีมงานลองสร้างโมเดลจากทุกชุดย่อยที่ไม่ว่างของตัวทำนายทั้ง 5 ตัว (ทุกโมเดลมี intercept, ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv`) แล้วเลือกโมเดลที่มี **AIC ต่ำที่สุด** ค่า **Adjusted $R^2$ ของโมเดลที่ถูกเลือก** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import itertools
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลลูกค้า
cust = pd.read_csv('biz_customers.csv')
candidates = ['income_k', 'tenure_months', 'loyalty_member', 'age', 'email_opens']

# 2. วนลูปทดสอบทุกชุดย่อยที่ไม่ว่าง (2^5 - 1 = 31 โมเดล)
best_aic = float('inf')
best_subset = None
best_model = None

for k in range(1, len(candidates) + 1):
    for subset in itertools.combinations(candidates, k):
        formula = 'spend_k ~ ' + ' + '.join(subset)
        m = smf.ols(formula, data=cust).fit()
        if m.aic < best_aic:
            best_aic = m.aic
            best_subset = subset
            best_model = m

print(f"ชุดตัวทำนายที่ให้ AIC ต่ำที่สุด: {best_subset}")
print(f"AIC ต่ำสุด: {best_aic:.4f}")
print(f"Adjusted R2 ของโมเดลที่ดีที่สุด: {best_model.rsquared_adj:.4f}")
print(f"คำตอบข้อ 9.6: {best_model.rsquared_adj:.4f}")


ชุดตัวทำนายที่ให้ AIC ต่ำที่สุด: ('income_k', 'tenure_months', 'loyalty_member', 'age')
AIC ต่ำสุด: 4167.3146
Adjusted R2 ของโมเดลที่ดีที่สุด: 0.2355
คำตอบข้อ 9.6: 0.2355


**โจทย์ข้อ 9.7:** กรณีศึกษานักวิเคราะห์ของแพลตฟอร์มอีคอมเมิร์ซต้องการทำนายยอดซื้อต่อเดือน (`spend_k`) และเริ่มจากโมเดลเต็มที่มีตัวทำนาย รายได้ (`income_k`), การเป็นสมาชิกสะสมแต้ม (`loyalty_member`), อายุ (`age`), จำนวนสินค้าเฉลี่ยต่อออเดอร์ (`basket_size`), จำนวนอีเมลโปรโมชันที่เปิดอ่าน (`email_opens`), จำนวนออเดอร์ใน 90 วัน (`orders_90d`) (มี intercept, ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv`) จากนั้นทำ **backward elimination**: ในแต่ละรอบถ้ามีตัวทำนายที่ $p\text{-value} > 0.05$ ให้ตัดตัวที่มี $p\text{-value}$ สูงที่สุดออกครั้งละ 1 ตัว แล้ว fit ใหม่ ทำซ้ำจนทุกตัวที่เหลือมี $p\text{-value} \le 0.05$ ค่า **AIC ของโมเดลสุดท้าย** (ค่า `aic` ที่ statsmodels รายงาน) = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. เริ่มต้นด้วยตัวทำนายทั้ง 6 ตัว
cust = pd.read_csv('biz_customers.csv')
current_features = ['income_k', 'loyalty_member', 'age', 'basket_size', 'email_opens', 'orders_90d']

# 2. ทำ Backward Elimination วนลูปตัดตัวแปรที่ p-value > 0.05
step = 1
while True:
    formula = 'spend_k ~ ' + ' + '.join(current_features)
    model = smf.ols(formula, data=cust).fit()
    # ดึง p-values ไม่รวม Intercept
    pvals = model.pvalues.drop('Intercept')
    max_p = pvals.max()
    worst_var = pvals.idxmax()
    
    print(f"รอบที่ {step}: ตัวแปรสูงสุดคือ {worst_var} (p-value = {max_p:.4f})")
    if max_p > 0.05:
        print(f"-> ตัด {worst_var} ออก")
        current_features.remove(worst_var)
        step += 1
    else:
        print("-> ทุกตัวแปรมี p-value <= 0.05 ยุติกระบวนการ")
        break

print(f"\nตัวแปรในโมเดลสุดท้าย: {current_features}")
print(f"ค่า AIC ของโมเดลสุดท้าย: {model.aic:.2f}")
print(f"คำตอบข้อ 9.7: {model.aic:.2f}")


รอบที่ 1: ตัวแปรสูงสุดคือ basket_size (p-value = 0.7068)
-> ตัด basket_size ออก
รอบที่ 2: ตัวแปรสูงสุดคือ email_opens (p-value = 0.6240)
-> ตัด email_opens ออก
รอบที่ 3: ตัวแปรสูงสุดคือ age (p-value = 0.0033)
-> ทุกตัวแปรมี p-value <= 0.05 ยุติกระบวนการ

ตัวแปรในโมเดลสุดท้าย: ['income_k', 'loyalty_member', 'age', 'orders_90d']
ค่า AIC ของโมเดลสุดท้าย: 4149.47
คำตอบข้อ 9.7: 4149.47


**โจทย์ข้อ 9.8:** กรณีศึกษาธนาคารที่ออกบัตรเครดิตร่วมกับร้านค้าปลีกต้องการประมาณยอดซื้อต่อเดือนเฉลี่ยของลูกค้าทุกคนที่มีลักษณะเดียวกับกลุ่มหนึ่ง คือ รายได้ (`income_k`) = 25 พันบาทต่อเดือน, จำนวนครั้งที่เข้าเว็บ/แอป (`web_visits`) = 9 ครั้ง, ระยะเวลาที่เป็นลูกค้า (`tenure_months`) = 31 เดือน จึง fit โมเดลทำนาย `spend_k` จาก `income_k`, `web_visits`, `tenure_months` (มี intercept, ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv`) **ขอบล่างของช่วงความเชื่อมั่น 95% ของยอดซื้อเฉลี่ยของกลุ่มนี้** (confidence interval ของค่าเฉลี่ย ไม่ใช่ของลูกค้ารายบุคคล) = ? พันบาทต่อเดือน (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดล
cust = pd.read_csv('biz_customers.csv')
formula = 'spend_k ~ income_k + web_visits + tenure_months'
model = smf.ols(formula, data=cust).fit()

# 2. กำหนดค่าโปรไฟล์ลูกค้ากลุ่มเป้าหมาย
target_profile = pd.DataFrame([{'income_k': 25, 'web_visits': 9, 'tenure_months': 31}])

# 3. คำนวณช่วงความเชื่อมั่นของค่าเฉลี่ย (Confidence Interval of the Mean)
pred_summary = model.get_prediction(target_profile).summary_frame(alpha=0.05)
print(pred_summary[['mean', 'mean_ci_lower', 'mean_ci_upper']])

ci_lower = pred_summary['mean_ci_lower'].iloc[0]
print(f"ขอบล่างของช่วงความเชื่อมั่น 95% (ค่าเฉลี่ย): {ci_lower:.4f}")
print(f"คำตอบข้อ 9.8: {ci_lower:.4f}")


       mean  mean_ci_lower  mean_ci_upper
0  3.328874       3.247397       3.410351
ขอบล่างของช่วงความเชื่อมั่น 95% (ค่าเฉลี่ย): 3.2474
คำตอบข้อ 9.8: 3.2474


**โจทย์ข้อ 9.9:** กรณีศึกษาผู้จัดการแบรนด์เครื่องดื่มใช้โมเดลที่ทำนาย `sales` (ยอดขายของแคมเปญ พันชิ้น) จาก `tv`, `radio`, `social`, `email` (งบโฆษณาแต่ละสื่อ พันบาท; มี intercept, fit ด้วยแคมเปญทั้ง 240 แคมเปญใน `biz_campaigns.csv`) เพื่อคาดการณ์ยอดขายของแคมเปญใหม่ที่มี $\text{tv} = 253, \text{radio} = 48, \text{social} = 26, \text{email} = 27$ **ขอบบนของ 95% Prediction Interval ของ sales ของแคมเปญนี้** = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. Fit โมเดลแคมเปญ
camp = pd.read_csv('biz_campaigns.csv')
formula = 'sales ~ tv + radio + social + email'
model = smf.ols(formula, data=camp).fit()

# 2. กำหนดงบโฆษณาของแคมเปญใหม่
new_campaign = pd.DataFrame([{'tv': 253, 'radio': 48, 'social': 26, 'email': 27}])

# 3. คำนวณช่วงการพยากรณ์สำหรับค่าสังเกตรายตัว (Prediction Interval)
pred_summary = model.get_prediction(new_campaign).summary_frame(alpha=0.05)
print(pred_summary[['mean', 'obs_ci_lower', 'obs_ci_upper']])

pi_upper = pred_summary['obs_ci_upper'].iloc[0]
print(f"ขอบบนของ 95% Prediction Interval: {pi_upper:.2f}")
print(f"คำตอบข้อ 9.9: {pi_upper:.2f}")


        mean  obs_ci_lower  obs_ci_upper
0  34.540791     30.376731      38.70485
ขอบบนของ 95% Prediction Interval: 38.70
คำตอบข้อ 9.9: 38.70


**โจทย์ข้อ 9.10:** กรณีศึกษานักวิเคราะห์การตลาดทำนาย `spend_k` (ยอดซื้อต่อเดือน) ด้วย `basket_size` (จำนวนสินค้าเฉลี่ยต่อออเดอร์) ตัวเดียว (simple regression) แล้วสังเกตสัมประสิทธิ์ของ `basket_size` จากนั้นเพิ่ม รายได้ (`income_k`) เข้าไปเป็นตัวทำนายด้วย (multiple regression) พบว่าสัมประสิทธิ์ของ `basket_size` เปลี่ยนไปอย่างเห็นได้ชัด (ทุกโมเดลมี intercept, ใช้ลูกค้าทั้ง 1203 คนใน `biz_customers.csv`) **ค่าสัมประสิทธิ์ของ `basket_size` ในโมเดล multiple ลบด้วยค่าในโมเดล simple** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. อ่านข้อมูลลูกค้า
cust = pd.read_csv('biz_customers.csv')

# 2. โมเดล Simple Regression: spend_k ~ basket_size
model_simple = smf.ols('spend_k ~ basket_size', data=cust).fit()
coef_simple = model_simple.params['basket_size']

# 3. โมเดล Multiple Regression: spend_k ~ basket_size + income_k
model_mult = smf.ols('spend_k ~ basket_size + income_k', data=cust).fit()
coef_mult = model_mult.params['basket_size']

# 4. หาผลต่าง: coef(multiple) - coef(simple)
diff_coef = coef_mult - coef_simple
print(f"สัมประสิทธิ์ basket_size ใน Simple: {coef_simple:.4f}")
print(f"สัมประสิทธิ์ basket_size ใน Multiple: {coef_mult:.6f}")
print(f"ผลต่าง (Multiple - Simple): {diff_coef:.4f}")
print(f"คำตอบข้อ 9.10: {diff_coef:.4f}")


สัมประสิทธิ์ basket_size ใน Simple: 0.2837
สัมประสิทธิ์ basket_size ใน Multiple: 0.000035
ผลต่าง (Multiple - Simple): -0.2836
คำตอบข้อ 9.10: -0.2836


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 9

---

### 1. แบบจำลอง Multiple Linear Regression (MLR)

แบบจำลองการถดถอยเชิงเส้นพหุคูณขยายจากตัวแปรเดี่ยวสู่ตัวแปรทำนายหลายตัว:
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p + \epsilon$$

- **การแปลความหมายสัมประสิทธิ์:** $\beta_j$ หมายถึงค่าที่คาดหวังของ $Y$ ที่จะเปลี่ยนแปลงไปเมื่อ $X_j$ เพิ่มขึ้น 1 หน่วย **โดยที่ตัวแปรทำนายอื่นๆ ทั้งหมดคงที่ (Holding other predictors constant)**
- **Omitted Variable Bias:** หากมีตัวแปรที่แท้จริงแต่ถูกละเลย และสัมพันธ์กับตัวแปรที่อยู่ในโมเดล ค่าประมาณความชันจะถูกบิดเบือนไป (เช่น สัมประสิทธิ์ของ `basket_size` ที่ลดฮวบเมื่อใส่ `income_k` เข้ามา)

---

### 2. การวัดความพอดีของโมเดล: Adjusted $R^2$, AIC และ BIC

1. **Adjusted $R^2$:**
   $$\text{Adj } R^2 = 1 - \frac{\text{RSS} / (n - p - 1)}{\text{TSS} / (n - 1)}$$
   ต่างจาก $R^2$ ทั่วไปที่จะเพิ่มขึ้นเสมอเมื่อใส่ตัวแปรใหม่ โดย Adjusted $R^2$ มีบทลงโทษสำหรับจำนวนตัวแปร $p$ จึงจะเพิ่มขึ้นก็ต่อเมื่อตัวแปรใหม่ช่วยอธิบาย $Y$ ได้มากกว่าการสุ่มทั่วไป

2. **Akaike Information Criterion (AIC):**
   $$\text{AIC} = 2k - 2\ln(\hat{L}) \approx n \ln(\text{RSS}/n) + 2p$$
   ใช้เปรียบเทียบโมเดล ยิ่งมีค่าน้อยยิ่งดี

3. **Bayesian Information Criterion (BIC):**
   $$\text{BIC} = k\ln(n) - 2\ln(\hat{L}) \approx n \ln(\text{RSS}/n) + p \ln(n)$$
   ลงโทษจำนวนตัวแปรหนักกว่า AIC เมื่อ $n > 7$ ทำให้มีแนวโน้มเลือกโมเดลที่กะทัดรัด (Simpler models) มากกว่า

---

### 3. การคัดเลือกตัวแปร (Variable Selection)

- **Best Subset Selection:** ทดสอบโมเดลที่เป็นไปได้ทั้งหมด $2^p - 1$ ชุด แล้วเลือกชุดที่ดีที่สุดตามเกณฑ์ เช่น AIC ต่ำสุด
- **Backward Elimination:** เริ่มต้นจากโมเดลที่มีตัวแปรครบทุกตัว แล้วค่อยๆ ตัดตัวแปรที่มี $p\text{-value}$ สูงที่สุดออกทีละตัว จนกว่าทุกตัวแปรที่เหลือจะมี $p\text{-value} \le \alpha$ (เช่น $0.05$)

---

### 4. ความแตกต่างระหว่าง Confidence Interval และ Prediction Interval

- **Confidence Interval ของค่าเฉลี่ย (Mean CI):**
  - ประเมินช่วงของ **ค่าเฉลี่ยของประชากร** ณ จุด $X = x_0$ เช่น ค่าเฉลี่ยยอดซื้อของกลุ่มลูกค้าที่มีลักษณะเดียวกัน
  - ความกว้างของช่วงขึ้นอยู่กับความไม่แน่นอนในการประมาณพารามิเตอร์ $\hat{\beta}$ เท่านั้น

- **Prediction Interval ของค่าสังเกตเดี่ยว (Individual PI):**
  - พยากรณ์ค่าของ **ลูกค้ารายใหม่ หรือ แคมเปญใหม่เพียง 1 รายการ**
  - ความกว้างของช่วงต้องรวมทั้งความไม่แน่นอนของพารามิเตอร์และความผันแปรสุ่มของ Error term ($\sigma^2$):
    $$\text{Var}(\hat{y}_0 + \epsilon) = \text{Var}(\hat{y}_0) + \sigma^2$$
  - ดังนั้น Prediction Interval จึง**กว้างกว่า** Confidence Interval เสมอ